In [ ]:
!pip install torch torchvision scikit-learn matplotlib pillow

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Training.zip to Training.zip


In [ ]:
!unzip Training.zip

Archive:  Training.zip
   creating: Training/
  inflating: Training/notsmoking_0001.jpg  
  inflating: Training/notsmoking_0002.jpg  
  inflating: Training/notsmoking_0003.jpg  
  inflating: Training/notsmoking_0005.jpg  
  inflating: Training/notsmoking_0007.jpg  
  inflating: Training/notsmoking_0008.jpg  
  inflating: Training/notsmoking_0009.jpg  
  inflating: Training/notsmoking_0010.jpg  
  inflating: Training/notsmoking_0011.jpg  
  inflating: Training/notsmoking_0012.jpg  
  inflating: Training/notsmoking_0013.jpg  
  inflating: Training/notsmoking_0015.jpg  
  inflating: Training/notsmoking_0016.jpg  
  inflating: Training/notsmoking_0019.jpg  
  inflating: Training/notsmoking_0020.jpg  
  inflating: Training/notsmoking_0021.jpg  
  inflating: Training/notsmoking_0022.jpg  
  inflating: Training/notsmoking_0023.jpg  
  inflating: Training/notsmoking_0025.jpg  
  inflating: Training/notsmoking_0027.jpg  
  inflating: Training/notsmoking_0028.jpg  
  inflating: Training/notsmoki

In [ ]:
import torch

In [ ]:
!ls

'archive (1).zip'   sample_data   Training   Training.zip


In [ ]:
!ls Training

notsmoking_0001.jpg  notsmoking_0276.jpg  smoking_0001.jpg  smoking_0266.jpg
notsmoking_0002.jpg  notsmoking_0277.jpg  smoking_0002.jpg  smoking_0267.jpg
notsmoking_0003.jpg  notsmoking_0279.jpg  smoking_0003.jpg  smoking_0269.jpg
notsmoking_0005.jpg  notsmoking_0280.jpg  smoking_0004.jpg  smoking_0270.jpg
notsmoking_0007.jpg  notsmoking_0282.jpg  smoking_0005.jpg  smoking_0271.jpg
notsmoking_0008.jpg  notsmoking_0285.jpg  smoking_0007.jpg  smoking_0272.jpg
notsmoking_0009.jpg  notsmoking_0286.jpg  smoking_0008.jpg  smoking_0273.jpg
notsmoking_0010.jpg  notsmoking_0287.jpg  smoking_0010.jpg  smoking_0274.jpg
notsmoking_0011.jpg  notsmoking_0288.jpg  smoking_0011.jpg  smoking_0275.jpg
notsmoking_0012.jpg  notsmoking_0290.jpg  smoking_0012.jpg  smoking_0276.jpg
notsmoking_0013.jpg  notsmoking_0292.jpg  smoking_0013.jpg  smoking_0278.jpg
notsmoking_0015.jpg  notsmoking_0293.jpg  smoking_0014.jpg  smoking_0283.jpg
notsmoking_0016.jpg  notsmoking_0295.jpg  smoking_0015.jpg  smoking_0284.jpg

In [ ]:
import os
import shutil

# Create main dataset folder
os.makedirs("DATA_SET/Smokers", exist_ok=True)
os.makedirs("DATA_SET/Non-Smoker", exist_ok=True)

source_folder = "Training"

for filename in os.listdir(source_folder):
    src_path = os.path.join(source_folder, filename)

    if filename.startswith("smoking"):
        shutil.copy(src_path, "DATA_SET/Smokers")
    elif filename.startswith("notsmoking"):
        shutil.copy(src_path, "DATA_SET/Non-Smoker")

print("Dataset reorganized successfully.")

Dataset reorganized successfully.


In [ ]:
!ls DATA_SET

Non-Smoker  Smokers


In [ ]:
!ls DATA_SET/Smokers | head

smoking_0001.jpg
smoking_0002.jpg
smoking_0003.jpg
smoking_0004.jpg
smoking_0005.jpg
smoking_0007.jpg
smoking_0008.jpg
smoking_0010.jpg
smoking_0011.jpg
smoking_0012.jpg


In [ ]:
!ls DATA_SET/Non-Smoker | head

notsmoking_0001.jpg
notsmoking_0002.jpg
notsmoking_0003.jpg
notsmoking_0005.jpg
notsmoking_0007.jpg
notsmoking_0008.jpg
notsmoking_0009.jpg
notsmoking_0010.jpg
notsmoking_0011.jpg
notsmoking_0012.jpg


In [ ]:
DATA_DIR = "DATA_SET"

In [ ]:
# Configuration
DATA_DIR = "DATA_SET"   # Make sure this folder exists
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cpu


In [ ]:
import torchvision

In [ ]:
print(torch.__version__)

2.10.0+cpu


In [ ]:
from torchvision import transforms

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
from torchvision import datasets
from torch.utils.data import DataLoader, random_split

dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_data, val_data, test_data = random_split(
    dataset, [train_size, val_size, test_size]
)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, batch_size=1)

class_names = dataset.classes
print("Classes:", class_names)
print("Total images:", len(dataset))

Classes: ['Non-Smoker', 'Smokers']
Total images: 716


In [ ]:
from torchvision import models
import torch.nn as nn
import torch.optim as optim

model = models.efficientnet_b0(weights="DEFAULT")
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 1)

model = model.to(DEVICE)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print("Model loaded successfully")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 60.7MB/s]


Model loaded successfully


In [ ]:
def train_model():
    model.train()

    for epoch in range(EPOCHS):
        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.float().to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.size(0)

        acc = correct / total
        print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {running_loss:.4f} Accuracy: {acc:.4f}")

In [ ]:
from sklearn.metrics import classification_report

def evaluate():
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            outputs = model(images).squeeze()
            preds = (torch.sigmoid(outputs) > 0.5).float()

            y_true.append(labels.item())
            y_pred.append(preds.item())

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
train_model()

Epoch [1/10] Loss: 11.7966 Accuracy: 0.6364
Epoch [2/10] Loss: 9.4913 Accuracy: 0.8199
Epoch [3/10] Loss: 6.8851 Accuracy: 0.8899
Epoch [4/10] Loss: 4.9976 Accuracy: 0.9091
Epoch [5/10] Loss: 3.7743 Accuracy: 0.9318
Epoch [6/10] Loss: 2.8475 Accuracy: 0.9563
Epoch [7/10] Loss: 2.1480 Accuracy: 0.9633
Epoch [8/10] Loss: 1.7409 Accuracy: 0.9738
Epoch [9/10] Loss: 1.3577 Accuracy: 0.9790
Epoch [10/10] Loss: 0.8272 Accuracy: 0.9965


In [28]:
evaluate()


Classification Report:
              precision    recall  f1-score   support

  Non-Smoker       0.85      0.87      0.86        38
     Smokers       0.85      0.83      0.84        35

    accuracy                           0.85        73
   macro avg       0.85      0.85      0.85        73
weighted avg       0.85      0.85      0.85        73



In [29]:
torch.save(model.state_dict(), "smoker_classifier.pth")
print("Model saved successfully.")

Model saved successfully.
